# SL-1b — Apprentissage PAC formellement : le lake `learning_theory_lean` exécuté en noyau Lean

Ce notebook est le **jumeau natif** de [SL-1 — Logical Learning](SL-1-LogicalLearning.ipynb) : là où SL-1 présente la série en Python, ici c'est le **noyau Lean 4 lui-même** qui parle. Chaque `#check` ci-dessous est exécuté par le kernel `lean4-wsl` **dans** le lake `learning_theory_lean` (dossier `ML/learning_theory_lean/`) — les signatures affichées sont celles que le compilateur a réellement vérifiées, pas des extraits copiés.

**Ce que le lake contient** : la théorie PAC (*Probably Approximately Correct*) et la convergence du perceptron, formalisées sur Mathlib v4.32.1 — erreur vraie, échantillon, borne de généralisation pour une classe finie, borne agnostique, concentration de Hoeffding-Chernoff, et le théorème de convergence du perceptron avec son contre-exemple de serrage (*tightness*). C'est le socle formel de la série SymbolicLearning.

**Prérequis** : kernel `lean4-wsl` (cf. `Lean-1-Setup.ipynb`) ; le kernel doit être lancé depuis le répertoire du lake (`ML/learning_theory_lean/`) pour que ses imports se résolvent — c'est le cas dans ce notebook.

**Relation au compagnon ML** : le lake a déjà un premier compagnon côté série ML — [2.8b-Theorie-PAC-Lean.ipynb](../../ML/DataScienceWithAgents/02-ML-Cours/2.8b-Theorie-PAC-Lean.ipynb), qui serre la main du modèle (distribution, échantillon, concentration uniforme). SL-1b va plus loin et plus large : la chaîne complète des bornes (ERM, union bound, Valiant classe finie, agnostique, Hoeffding-Chernoff) et toute la branche Perceptron (Novikoff + serrage), avec exercices.

In [1]:
import PacLearning_en
import PacLearning.ERM
import PacLearning.UniformConcentration
import PacLearning.Agnostic
import Perceptron_en

import PacLearning_en
import PacLearning.ERM
import PacLearning.UniformConcentration
import PacLearning.Agnostic
import Perceptron_en
--% env 0

Raw input:
{"cmd": "import PacLearning_en\nimport PacLearning.ERM\nimport PacLearning.UniformConcentration\nimport PacLearning.Agnostic\nimport Perceptron_en"}
Raw output:
{"env": 0}

In [2]:
#eval 2 + 2

#eval 2 + 2
─────▶  4
--% env 1

Raw input:
{"cmd": "#eval 2 + 2", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 5},
   "data": "4"}],
 "env": 1}

## 1. Le vocabulaire PAC : distribution, hypothèse, erreur vraie

Le lake évite délibérément la machinerie `ℝ≥0∞`/`Measure` de Mathlib : une **distribution** sur un type fini `X` est une fonction de poids `X → ℝ`, positive, de masse totale 1 (`PacLearning/Data.lean`). Une **hypothèse** est un étiqueteur booléen `X → Bool`, et l'**erreur vraie** de `h` contre le concept cible `f` est la masse des instances mal classées.

In [3]:
#check PacLearning.Distribution
#check PacLearning.Hypothesis
#check PacLearning.trueError

#check PacLearning.trueError_nonneg
#check PacLearning.trueError_self
#check PacLearning.trueError_le_one
#check PacLearning.trueError_comm

#check PacLearning.Distribution
──────▶  PacLearning.Distribution.{u_2} (X : Type u_2) [Fintype X] : Type u_2
#check PacLearning.Hypothesis
──────▶  PacLearning.Hypothesis.{u_2} (X : Type u_2) : Type u_2
#check PacLearning.trueError
──────▶  PacLearning.trueError.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X)
  (f h : PacLearning.Hypothesis X) : ℝ

#check PacLearning.trueError_nonneg
──────▶  PacLearning.trueError_nonneg.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  {f h : PacLearning.Hypothesis X} : 0 ≤ PacLearning.trueError D f h
#check PacLearning.trueError_self
──────▶  PacLearning.trueError_self.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  {f : PacLearning.Hypothesis X} : PacLearning.trueError D f f = 0
#check PacLearning.trueError_le_one
──────▶  PacLearning.trueError_le_one.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  {f h : PacLearning.Hypothesis X} : PacLearning.trueError D f h ≤ 1
#check PacLearning.trueError_comm
──────▶  PacLearning.trueError_comm.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  {f h : PacLearning.Hypothesis X} : PacLearning.trueError D f h = PacLearning.trueError D h f
--% env 2

Raw input:
{"cmd": "#check PacLearning.Distribution\n#check PacLearning.Hypothesis\n#check PacLearning.trueError\n\n#check PacLearning.trueError_nonneg\n#check PacLearning.trueError_self\n#check PacLearning.trueError_le_one\n#check PacLearning.trueError_comm", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.Distribution.{u_2} (X : Type u_2) [Fintype X] : Type u_2"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "PacLearning.Hypothesis.{u_2} (X : Type u_2) : Type u_2"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "PacLearning.trueError.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X)\n  (f h : PacLearning.Hypothesis X) : ℝ"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "PacLearning.trueError_nonneg.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  {f h : PacLearning.Hypothesis X} : 0 ≤ PacLearning.trueError D f h"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "PacLearning.trueError_self.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  {f : PacLearning.Hypothesis X} : PacLearning.trueError D f f = 0"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "PacLearning.trueError_le_one.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  {f h : PacLearning.Hypothesis X} : PacLearning.trueError D f h ≤ 1"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "PacLearning.trueError_comm.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  {f h : PacLearning.Hypothesis X} : PacLearning.trueError D f h = PacLearning.trueError D h f"}],
 "env": 2}

Une distribution concrète, construite à la main sur deux instances — uniforme, chaque poids vaut un demi :

In [4]:
noncomputable def Dcoin : PacLearning.Distribution (Fin 2) where
  weight := fun _ => 1 / 2
  nonneg := by intro x; norm_num
  sum_one := by simp

#check Dcoin

noncomputable def Dcoin : PacLearning.Distribution (Fin 2) where
  weight := fun _ => 1 / 2
  nonneg := by intro x; norm_num
  sum_one := by simp

#check Dcoin
──────▶  Dcoin : PacLearning.Distribution (Fin 2)
--% env 3

Raw input:
{"cmd": "noncomputable def Dcoin : PacLearning.Distribution (Fin 2) where\n  weight := fun _ => 1 / 2\n  nonneg := by intro x; norm_num\n  sum_one := by simp\n\n#check Dcoin", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "Dcoin : PacLearning.Distribution (Fin 2)"}],
 "env": 3}

## 2. L'échantillon : tirages i.i.d. et poids d'un échantillon

Un échantillon de taille `n` est une fonction `Fin n → X` ; son poids sous `D` est le produit des poids de ses instances (`PacLearning/Sample.lean`). Le théorème `sampleWeight_sum_one` dit que les échantillons forment eux-mêmes une distribution — c'est la pierre d'angle de toute la théorie PAC : raisonner sur « l'apprentissage réussit sur un tirage aléatoire ».

In [5]:
#check PacLearning.sampleWeight
#check PacLearning.sampleWeight_nonneg
#check PacLearning.sampleWeight_sum_one

#check PacLearning.sampleWeight
──────▶  PacLearning.sampleWeight.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X) {n : ℕ} (S : Fin n → X) : ℝ
#check PacLearning.sampleWeight_nonneg
──────▶  PacLearning.sampleWeight_nonneg.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  (S : Fin n → X) : 0 ≤ PacLearning.sampleWeight D S
#check PacLearning.sampleWeight_sum_one
──────▶  PacLearning.sampleWeight_sum_one.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} (n : ℕ) :
  ∑ S, PacLearning.sampleWeight D S = 1
--% env 4

Raw input:
{"cmd": "#check PacLearning.sampleWeight\n#check PacLearning.sampleWeight_nonneg\n#check PacLearning.sampleWeight_sum_one", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.sampleWeight.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X) {n : ℕ} (S : Fin n → X) : ℝ"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "PacLearning.sampleWeight_nonneg.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}\n  (S : Fin n → X) : 0 ≤ PacLearning.sampleWeight D S"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "PacLearning.sampleWeight_sum_one.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} (n : ℕ) :\n  ∑ S, PacLearning.sampleWeight D S = 1"}],
 "env": 4}

## 3. La borne de généralisation pour une classe finie

Le résultat central : pour une **classe finie** d'hypothèses, l'ERM (*Empirical Risk Minimization*) généralise — si l'échantillon est assez grand (en `log |H|`), l'erreur empirique approche l'erreur vraie uniformément sur la classe. La chaîne se lit dans les modules : `ERM` borne l'écart empirical/vrai du minimiseur empirique, `UniformConcentration` en donne la version **uniforme sur toute la classe** (là où Hoeffding seul ne contrôlerait qu'une hypothèse fixée), `UnionBound` fait entrer la finitude de la classe dans le contrôle, et `PacFiniteBound` assemble le tout dans la borne PAC complète.

In [6]:
#check PacLearning.erm_error_bound
#check PacLearning.uniform_concentration
#check PacLearning.sampleProb_union_bound
#check PacLearning.pac_finite_class_bound_aux
#check PacLearning.pac_finite_class_bound
#check PacLearning.one_sub_pow_le_exp
#check PacLearning.empError_eq_zero_iff

#check PacLearning.erm_error_bound
──────▶  PacLearning.erm_error_bound.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) (S : Fin n → X) {ε : ℝ}
  (hε : 0 < ε) (ĥ hOpt : PacLearning.Hypothesis X) (hĥ_mem : ĥ ∈ Hs) (hOpt_mem : hOpt ∈ Hs)
  (hconc : ∀ h ∈ Hs, |PacLearning.empError f h S - PacLearning.trueError D f h| ≤ ε)
  (hĥ_erm : ∀ h ∈ Hs, PacLearning.empError f ĥ S ≤ PacLearning.empError f h S) :
  PacLearning.trueError D f ĥ ≤ PacLearning.trueError D f hOpt + 2 * ε
#check PacLearning.uniform_concentration
──────▶  PacLearning.uniform_concentration.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) {ε : ℝ} (hε : 0 < ε) :
  (PacLearning.sampleProb D fun S => ∃ h ∈ Hs, ε ≤ |PacLearning.empError f h S - PacLearning.trueError D f h|) ≤
    ↑Hs.card * (2 * Real.exp (-(2 * ↑n * ε ^ 2)))
#check PacLearning.sampleProb_union_bound
──────▶  PacLearning.sampleProb_union_bound.{u_1, u_2} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  {ι : Type u_2} [Fintype ι] (s : Finset ι) [DecidableEq ι] (P : ι → (Fin n → X) → Prop) :
  (PacLearning.sampleProb D fun S => ∃ i ∈ s, P i S) ≤ ∑ i ∈ s, PacLearning.sampleProb D (P i)
#check PacLearning.pac_finite_class_bound_aux
──────▶  PacLearning.pac_finite_class_bound_aux.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X) {n : ℕ}
  (Hs : Finset (PacLearning.Hypothesis X)) (f : PacLearning.Hypothesis X) (ε δ : ℝ) (hε : 0 < ε) (hδ : 0 < δ)
  (hH : 0 < ↑Hs.card) (hn : 0 < n) (hm : 1 / ε * (Real.log ↑Hs.card + Real.log (1 / δ)) ≤ ↑n)
  (hDec : DecidablePred fun S => ∃ hyp ∈ Hs, PacLearning.empError f hyp S = 0 ∧ ε < PacLearning.trueError D f hyp) :
  (PacLearning.sampleProb D fun S => ∃ hyp ∈ Hs, PacLearning.empError f hyp S = 0 ∧ ε < PacLearning.trueError D f hyp) ≤
    δ
#check PacLearning.pac_finite_class_bound
──────▶  PacLearning.pac_finite_class_bound.{u_1} {X : Type u_1} [Fintype X] (D : PacLearning.Distribution X) {n : ℕ}
  (Hs : Finset (PacLearning.Hypothesis X)) (f : PacLearning.Hypothesis X) (ε δ : ℝ) (hε : 0 < ε) (hδ : 0 < δ)
  (hH : 0 < ↑Hs.card) (hn : 0 < n) (hm : 1 / ε * (Real.log ↑Hs.card + Real.log (1 / δ)) ≤ ↑n) :
  (PacLearning.sampleProb D fun S => ∃ hyp ∈ Hs, PacLearning.empError f hyp S = 0 ∧ ε < PacLearning.trueError D f hyp) ≤
    δ
#check PacLearning.one_sub_pow_le_exp
──────▶  PacLearning.one_sub_pow_le_exp (x : ℝ) (n : ℕ) (hx0 : 0 ≤ x) (hx1 : x ≤ 1) (ε : ℝ) (hxe : ε ≤ x) :
  (1 - x) ^ n ≤ Real.exp (-(ε * ↑n))
#check PacLearning.empError_eq_zero_iff
──────▶  PacLearning.empError_eq_zero_iff.{u_1} {X : Type u_1} [Fintype X] {n : ℕ} (f hyp : PacLearning.Hypothesis X)
  (S : Fin n → X) (hn : 0 < n) : PacLearning.empError f hyp S = 0 ↔ ∀ (i : Fin n), hyp (S i) = f (S i)
--% env 5

Raw input:
{"cmd": "#check PacLearning.erm_error_bound\n#check PacLearning.uniform_concentration\n#check PacLearning.sampleProb_union_bound\n#check PacLearning.pac_finite_class_bound_aux\n#check PacLearning.pac_finite_class_bound\n#check PacLearning.one_sub_pow_le_exp\n#check PacLearning.empError_eq_zero_iff", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.erm_error_bound.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) (S : Fin n → X) {ε : ℝ}\n  (hε : 0 < ε) (ĥ hOpt : PacLearning.Hypothesis X) (hĥ_mem : ĥ ∈ Hs) (hOpt_mem : hOpt ∈ Hs)\n  (hconc : ∀ h ∈ Hs, |PacLearning.empError f h S - PacLearning.trueError D f h| ≤ ε)\n  (hĥ_erm : ∀ h ∈ Hs, PacLearning.empError f ĥ S ≤ PacLearning.empError f h S) :\n  PacLearning.trueError D f ĥ ≤ PacLearning.trueError D f hOpt + 2 * ε"},
  {"severity": "info",
   "pos": {"line": 2, "colum

## 4. Le cadre agnostique : pas de concept cible atteignable

En agnostique, aucune hypothèse de la classe ne réalise l'erreur nulle — on compare au **meilleur de la classe**. Le module `Agnostic` montre que la même borne tient relativement à l'optimum de la classe (`sampleProb_mono` en est la brique : si un événement en couvre un autre, sa probabilité est plus petite).

In [7]:
#check PacLearning.sampleProb_mono
#check PacLearning.pac_agnostic_generalization

#check PacLearning.sampleProb_mono
──────▶  PacLearning.sampleProb_mono.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  (P Q : (Fin n → X) → Prop) [DecidablePred P] [DecidablePred Q] (h : ∀ (S : Fin n → X), P S → Q S) :
  PacLearning.sampleProb D P ≤ PacLearning.sampleProb D Q
#check PacLearning.pac_agnostic_generalization
──────▶  PacLearning.pac_agnostic_generalization.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) {ε : ℝ} (hε : 0 < ε)
  (ĥ : (Fin n → X) → PacLearning.Hypothesis X) (hOpt : PacLearning.Hypothesis X) (hOpt_mem : hOpt ∈ Hs)
  (hĥ_mem : ∀ (S : Fin n → X), ĥ S ∈ Hs)
  (hĥ_erm : ∀ (S : Fin n → X), ∀ h ∈ Hs, PacLearning.empError f (ĥ S) S ≤ PacLearning.empError f h S) :
  1 - ↑Hs.card * (2 * Real.exp (-(2 * ↑n * ε ^ 2))) ≤
    PacLearning.sampleProb D fun S => PacLearning.trueError D f (ĥ S) ≤ PacLearning.trueError D f hOpt + 2 * ε
--% env 6

Raw input:
{"cmd": "#check PacLearning.sampleProb_mono\n#check PacLearning.pac_agnostic_generalization", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.sampleProb_mono.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}\n  (P Q : (Fin n → X) → Prop) [DecidablePred P] [DecidablePred Q] (h : ∀ (S : Fin n → X), P S → Q S) :\n  PacLearning.sampleProb D P ≤ PacLearning.sampleProb D Q"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "PacLearning.pac_agnostic_generalization.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  (f : PacLearning.Hypothesis X) (Hs : Finset (PacLearning.Hypothesis X)) {n : ℕ} (hn : 0 < n) {ε : ℝ} (hε : 0 < ε)\n  (ĥ : (Fin n → X) → PacLearning.Hypothesis X) (hOpt : PacLearning.Hypothesis X) (hOpt_mem : hOpt ∈ Hs)\n  (hĥ_mem : ∀ (S : Fin n → X), ĥ S ∈ Hs)\n  (hĥ_erm : ∀ (S : Fin n → X), ∀ h ∈ Hs, PacLearning.empError f (ĥ S) S ≤ PacLearning.empError f h S) :\n  1 - ↑Hs.card * (2 * Real.exp (-(2 * ↑n * ε ^ 2))) ≤\n    PacLearning.sampleProb D fun S => PacLearning.trueError D f (ĥ S) ≤ PacLearning.trueError D f hOpt + 2 * ε"}],
 "env": 6}

## 5. La concentration : de Markov à Hoeffding

La machinerie probabiliste descend l'échelle classique des inégalités de concentration : `Concentration` pose l'espérance discrète et l'inégalité de **Markov**, `Hoeffding` en tire Chernoff (par fonction génératrice des moments) puis les deux queues et la borne de concentration bilatérale ; `SampleExpect` fournit l'espérance sur l'espace des échantillons — dont le jalon `sampleExpect_empError_eq_trueError` : *l'erreur empirique est un estimateur sans biais de l'erreur vraie*. Le cœur calculatoire de la dérivation (`MGF`, `BernoulliMGF` : dérivées log-MGF, moyennes basculées) n'est pas visité cellule par cellule ici — c'est l'objet du README du lake.

In [8]:
#check PacLearning.markov_ineq
#check PacLearning.chernoff_ineq
#check PacLearning.hoeffding_mgf_sum_le
#check PacLearning.hoeffding_upper_tail
#check PacLearning.hoeffding_concentration
#check PacLearning.sampleExpect_empError_eq_trueError
#check PacLearning.sampleExpect_mul_const

#check PacLearning.markov_ineq
──────▶  PacLearning.markov_ineq.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {g : X → ℝ}
  (hg : ∀ (x : X), 0 ≤ g x) {t : ℝ} (ht : 0 < t) : ∑ x with t ≤ g x, D.weight x ≤ PacLearning.expect D g / t
#check PacLearning.chernoff_ineq
──────▶  PacLearning.chernoff_ineq.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  (Y : (Fin n → X) → ℝ) (a t : ℝ) (ht : 0 < t) :
  (PacLearning.sampleProb D fun S => a ≤ Y S) ≤
    (PacLearning.sampleExpect D fun S => Real.exp (t * Y S)) * Real.exp (-(t * a))
#check PacLearning.hoeffding_mgf_sum_le
──────▶  PacLearning.hoeffding_mgf_sum_le.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f h : PacLearning.Hypothesis X) {n : ℕ} (t : ℝ) :
  (PacLearning.sampleExpect D fun S =>
      Real.exp (t * ∑ i, ((if h (S i) ≠ f (S i) then 1 else 0) - PacLearning.trueError D f h))) ≤
    Real.exp (↑n * t ^ 2 / 8)
#check PacLearning.hoeffding_upper_tail
──────▶  PacLearning.hoeffding_upper_tail.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f h : PacLearning.Hypothesis X) {n : ℕ} {ε : ℝ} (hε : 0 < ε) :
  (PacLearning.sampleProb D fun S =>
      ↑n * ε ≤ ∑ i, ((if h (S i) ≠ f (S i) then 1 else 0) - PacLearning.trueError D f h)) ≤
    Real.exp (-(2 * ↑n * ε ^ 2))
#check PacLearning.hoeffding_concentration
──────▶  PacLearning.hoeffding_concentration.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}
  (f h : PacLearning.Hypothesis X) {n : ℕ} (hn : 0 < n) {ε : ℝ} (hε : 0 < ε) :
  (PacLearning.sampleProb D fun S => ε ≤ |PacLearning.empError f h S - PacLearning.trueError D f h|) ≤
    2 * Real.exp (-(2 * ↑n * ε ^ 2))
#check PacLearning.sampleExpect_empError_eq_trueError
──────▶  PacLearning.sampleExpect_empError_eq_trueError.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}
  (f h : PacLearning.Hypothesis X) (hn : 0 < n) :
  (PacLearning.sampleExpect D fun S => PacLearning.empError f h S) = PacLearning.trueError D f h
#check PacLearning.sampleExpect_mul_const
──────▶  PacLearning.sampleExpect_mul_const.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ} (c : ℝ)
  (g : (Fin n → X) → ℝ) : (PacLearning.sampleExpect D fun S => g S * c) = PacLearning.sampleExpect D g * c
--% env 7

Raw input:
{"cmd": "#check PacLearning.markov_ineq\n#check PacLearning.chernoff_ineq\n#check PacLearning.hoeffding_mgf_sum_le\n#check PacLearning.hoeffding_upper_tail\n#check PacLearning.hoeffding_concentration\n#check PacLearning.sampleExpect_empError_eq_trueError\n#check PacLearning.sampleExpect_mul_const", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "PacLearning.markov_ineq.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {g : X → ℝ}\n  (hg : ∀ (x : X), 0 ≤ g x) {t : ℝ} (ht : 0 < t) : ∑ x with t ≤ g x, D.weight x ≤ PacLearning.expect D g / t"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "PacLearning.chernoff_ineq.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X} {n : ℕ}\n  (Y : (Fin n → X) → ℝ) (a t : ℝ) (ht : 0 < t) :\n  (PacLearning.sampleProb D fun S => a ≤ Y S) ≤\n    (PacLearning.sampleExpect D fun S => Real.exp (t * Y S)) * Real.exp (-(t * a))"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "PacLearning.hoeffding_mgf_sum_le.{u_1} {X : Type u_1} [Fintype X] {D : PacLearning.Distribution X}\n  (f h : PacLearning.Hypothesis X) {n : ℕ} (t : ℝ) :\n  (PacLearning.sampleExpect D fun S =>\n      Real.exp (t * ∑ i, ((if h (S i) ≠ f (S i) then 1 else 0) - PacLearning.trueError D f h))) ≤\n    Real.exp (↑n * t ^ 2 / 8)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "PacLearning.hoeffding_upper_tail.{u_1} {X : Type

## 6. Le perceptron : convergence et serrage

Second pilier du lake (`Perceptron/`) : l'algorithme du perceptron en espace de Hilbert réel. `perceptronWeights_zero`/`_succ` définissent la trajectoire des poids par récursion sur les erreurs. Le module `Convergence` porte la **borne de Novikoff** : croissance de l'alignement (`align_growth`), contrôle de la norme (`norm_bound`), d'où le plafond d'erreurs en `R²/γ²` (`novikoff_mistake_bound`). Et `Tightness` construit le contre-exemple qui montre que cette borne est **serrée** — des témoins explicites (`witnessPts`, `witnessLbl`) pour lesquels l'algorithme fait exactement le nombre d'erreurs annoncé, jusqu'au théorème final `novikoff_bound_is_sharp`.

In [9]:
#check Perceptron.IsLabel
#check Perceptron.norm_sq_eq_inner_self
#check Perceptron.perceptronWeights_zero
#check Perceptron.perceptronWeights_succ
#check Perceptron.PerceptronRun.align_growth
#check Perceptron.PerceptronRun.norm_bound
#check Perceptron.PerceptronRun.novikoff_mistake_bound
#check Perceptron.witnessPts
#check Perceptron.witnessLbl
#check Perceptron.witness_margin_inner
#check Perceptron.novikoff_bound_is_sharp

#check Perceptron.IsLabel
──────▶  Perceptron.IsLabel (y : ℝ) : Prop
#check Perceptron.norm_sq_eq_inner_self
──────▶  Perceptron.norm_sq_eq_inner_self.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (x : V) :
  ‖x‖ ^ 2 = inner ℝ x x
#check Perceptron.perceptronWeights_zero
──────▶  Perceptron.perceptronWeights_zero.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (pts : ℕ → V)
  (lbl : ℕ → ℝ) : Perceptron.perceptronWeights pts lbl 0 = 0
#check Perceptron.perceptronWeights_succ
──────▶  Perceptron.perceptronWeights_succ.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (pts : ℕ → V)
  (lbl : ℕ → ℝ) (k : ℕ) :
  Perceptron.perceptronWeights pts lbl (k + 1) = Perceptron.perceptronWeights pts lbl k + lbl k • pts k
#check Perceptron.PerceptronRun.align_growth
──────▶  Perceptron.PerceptronRun.align_growth.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V]
  (run : Perceptron.PerceptronRun V) (k : ℕ) :
  k ≤ run.n → ↑k * run.γ ≤ inner ℝ (Perceptron.perceptronWeights run.pts run.lbl k) run.u
#check Perceptron.PerceptronRun.norm_bound
──────▶  Perceptron.PerceptronRun.norm_bound.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V]
  (run : Perceptron.PerceptronRun V) (k : ℕ) :
  k ≤ run.n → ‖Perceptron.perceptronWeights run.pts run.lbl k‖ ^ 2 ≤ ↑k * run.R ^ 2
#check Perceptron.PerceptronRun.novikoff_mistake_bound
──────▶  Perceptron.PerceptronRun.novikoff_mistake_bound.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V]
  (run : Perceptron.PerceptronRun V) : ↑run.n * run.γ ^ 2 ≤ run.R ^ 2
#check Perceptron.witnessPts
──────▶  Perceptron.witnessPts : ℕ → ℂ
#check Perceptron.witnessLbl
──────▶  Perceptron.witnessLbl : ℕ → ℝ
#check Perceptron.witness_margin_inner
──────▶  Perceptron.witness_margin_inner (k : ℕ) : inner ℝ 1 (Perceptron.witnessPts k) = 1
#check Perceptron.novikoff_bound_is_sharp
──────▶  Perceptron.novikoff_bound_is_sharp : ∃ run, ↑run.n * run.γ ^ 2 = run.R ^ 2
--% env 8

Raw input:
{"cmd": "#check Perceptron.IsLabel\n#check Perceptron.norm_sq_eq_inner_self\n#check Perceptron.perceptronWeights_zero\n#check Perceptron.perceptronWeights_succ\n#check Perceptron.PerceptronRun.align_growth\n#check Perceptron.PerceptronRun.norm_bound\n#check Perceptron.PerceptronRun.novikoff_mistake_bound\n#check Perceptron.witnessPts\n#check Perceptron.witnessLbl\n#check Perceptron.witness_margin_inner\n#check Perceptron.novikoff_bound_is_sharp", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "Perceptron.IsLabel (y : ℝ) : Prop"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Perceptron.norm_sq_eq_inner_self.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (x : V) :\n  ‖x‖ ^ 2 = inner ℝ x x"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Perceptron.perceptronWeights_zero.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (pts : ℕ → V)\n  (lbl : ℕ → ℝ) : Perceptron.perceptronWeights pts lbl 0 = 0"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "Perceptron.perceptronWeights_succ.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V] (pts : ℕ → V)\n  (lbl : ℕ → ℝ) (k : ℕ) :\n  Perceptron.perceptronWeights pts lbl (k + 1) = Perceptron.perceptronWeights pts lbl k + lbl k • pts k"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "Perceptron.PerceptronRun.align_growth.{u_1} {V : Type u_1} [SeminormedAddCommGroup V] [InnerProductSpace ℝ V]\n  (run : Perceptron.PerceptronRun V) (k : ℕ) :\n  k ≤ run.n → ↑k * run.γ ≤ inner ℝ (Perceptron.perceptronWeights run.pts run.lbl k) run.u"},
  {"sev

## 7. Lecture du fil

Ce que les signatures ci-dessus racontent, prises ensemble :

- **le modèle est autosuffisant** — `Distribution` est une structure de trois champs lisibles, pas un `Measure` de Mathlib ; un étudiant peut construire une distribution à la main (nous l'avons fait avec `Dcoin`) et la manipuler ;
- **la chaîne de dépendances est celle du cours** — vocabulaire (`Data`) → échantillon (`Sample`) → concentration (`MGF`/`BernoulliMGF`/`Hoeffding`) → union bound (`UnionBound`) → borne finie (`ERM`, `PacFiniteBound`) → agnostique (`Agnostic`), et en parallèle la branche géométrique du perceptron (`Perceptron/*`, `Convergence`, `Tightness`) ;
- **chaque constante chiffrée d'un manuel a son théorème** — `pac_finite_class_bound` porte le `log |H| + log(1/δ)` de la complexité d'échantillon, `chernoff_ineq` l'exponentielle de Hoeffding, et le théorème de convergence du perceptron sa borne en `1/γ²` ;
- **le serrage n'est pas un détail** — `Tightness` prouve que la borne perceptron n'est pas pessimiste : le contre-exemple `witnessPts`/`witnessLbl` fait exactement le nombre d'erreurs de la borne. C'est la différence entre « la preuve passe » et « la preuve dit quelque chose ».

C'est la promesse du compagnon natif : la visibilité du lake passe par le compilateur, pas par une transcription.

## Exercices

Les exercices suivants sont à compléter. Ils utilisent `Dcoin` (section 1) et les théorèmes `#check`-és ci-dessus. Remplacer chaque `sorry` par une preuve ; les indices sont dans les commentaires.

### Exercice 1 : erreur nulle contre soi-même

Prouver que l'hypothèse constante vraie fait une erreur vraie nulle contre elle-même, sous `Dcoin`.

*Indice* : c'est une spécialisation directe de `PacLearning.trueError_self`.

In [10]:
-- Exercice 1 : a completer
-- TODO etudiant
theorem exo1_self_zero :
    PacLearning.trueError Dcoin (fun _ => true) (fun _ => true) = 0 := by
  sorry

-- Exercice 1 : a completer
-- TODO etudiant
theorem exo1_self_zero :
        ──────────────▶ 🟨 declaration uses `sorry`
    PacLearning.trueError Dcoin (fun _ => true) (fun _ => true) = 0 := by
  sorry
--% env 9
--% prove 0

Raw input:
{"cmd": "-- Exercice 1 : a completer\n-- TODO etudiant\ntheorem exo1_self_zero :\n    PacLearning.trueError Dcoin (fun _ => true) (fun _ => true) = 0 := by\n  sorry", "env": 8}
Raw output:
{"sorries":
 [{"proofState": 0,
   "pos": {"line": 5, "column": 2},
   "goal": "⊢ (PacLearning.trueError Dcoin (fun x => true) fun x => true) = 0",
   "endPos": {"line": 5, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 22},
   "data": "declaration uses `sorry`"}],
 "env": 9}

### Exercice 2 : la masse des échantillons vaut un

Prouver que pour des échantillons de taille 1 sur `Fin 2`, la somme des poids vaut 1.

*Indice* : appliquer `PacLearning.sampleWeight_sum_one` — `D` y est implicite, seul `n` s'écrit explicitement.

In [11]:
-- Exercice 2 : a completer
-- TODO etudiant
theorem exo2_masse_un :
    ∑ S : Fin 1 → Fin 2, PacLearning.sampleWeight Dcoin S = 1 := by
  sorry

-- Exercice 2 : a completer
-- TODO etudiant
theorem exo2_masse_un :
        ─────────────▶ 🟨 declaration uses `sorry`
    ∑ S : Fin 1 → Fin 2, PacLearning.sampleWeight Dcoin S = 1 := by
  sorry
--% env 10
--% prove 1

Raw input:
{"cmd": "-- Exercice 2 : a completer\n-- TODO etudiant\ntheorem exo2_masse_un :\n    \u2211 S : Fin 1 \u2192 Fin 2, PacLearning.sampleWeight Dcoin S = 1 := by\n  sorry", "env": 9}
Raw output:
{"sorries":
 [{"proofState": 1,
   "pos": {"line": 5, "column": 2},
   "goal": "⊢ ∑ S, PacLearning.sampleWeight Dcoin S = 1",
   "endPos": {"line": 5, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 21},
   "data": "declaration uses `sorry`"}],
 "env": 10}

### Exercice 3 : symétrie du désaccord

Prouver que l'erreur de `h` contre `f` égale celle de `f` contre `h` (le désaccord est symétrique).

*Indice* : `PacLearning.trueError_comm`, appliqué dans le bon sens.

In [12]:
-- Exercice 3 : a completer
-- TODO etudiant
theorem exo3_symetrie (f h : PacLearning.Hypothesis (Fin 2)) :
    PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f := by
  sorry

-- Exercice 3 : a completer
-- TODO etudiant
theorem exo3_symetrie (f h : PacLearning.Hypothesis (Fin 2)) :
        ─────────────▶ 🟨 declaration uses `sorry`
    PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f := by
  sorry
--% env 11
--% prove 2

Raw input:
{"cmd": "-- Exercice 3 : a completer\n-- TODO etudiant\ntheorem exo3_symetrie (f h : PacLearning.Hypothesis (Fin 2)) :\n    PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f := by\n  sorry", "env": 10}
Raw output:
{"sorries":
 [{"proofState": 2,
   "pos": {"line": 5, "column": 2},
   "goal":
   "f h : PacLearning.Hypothesis (Fin 2)\n⊢ PacLearning.trueError Dcoin f h = PacLearning.trueError Dcoin h f",
   "endPos": {"line": 5, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 21},
   "data": "declaration uses `sorry`"}],
 "env": 11}

## Conclusion

Ce compagnon a parcouru **quatorze modules** du lake `learning_theory_lean` en exécutant leurs déclarations dans le noyau Lean : côté théorie PAC, `Data`, `Sample`, `SampleExpect`, `Concentration`, `Hoeffding`, `ERM`, `UniformConcentration`, `UnionBound`, `PacFiniteBound`, `Agnostic` ; côté géométrie, `Perceptron.Data`, `Perceptron`, `Convergence`, `Tightness`. Seuls `MGF` et `BernoulliMGF` — le cœur calculatoire de la dérivation de concentration — ne sont visités qu'en prose. Avant cette série de `#check`, la quasi-totalité de ces modules n'était citée par aucun notebook du dépôt : leur contenu formel existait pour le compilateur seul.

**Pour aller plus loin** :

- [SL-1 — Logical Learning](SL-1-LogicalLearning.ipynb) : la présentation Python de la série, figures à l'appui ;
- [2.8b-Theorie-PAC-Lean.ipynb](../../ML/DataScienceWithAgents/02-ML-Cours/2.8b-Theorie-PAC-Lean.ipynb) : le premier compagnon du lake, côté série ML (modèle et échantillon) ;
- le [README du lake](../../ML/learning_theory_lean/README.md) : la carte complète des modules, `MGF` et `BernoulliMGF` inclus ;
- [SL-2 — Knowledge-Based Learning](SL-2-KnowledgeBasedLearning.ipynb) : la suite de la série, du côté connaissance.

**Références** : Mohri, Rostamizadeh & Talwalkar, *Foundations of Machine Learning* (2e éd.), ch. 2-3 ; Shalev-Shwartz & Ben-David, *Understanding Machine Learning*, ch. 21 (perceptron).